# 05 · Review Titles

**What this does:** this is the **only** place where approved film details change: names, dates, the **In MMM** flag, and which media product codes and revenue product IDs belong to which film. Decisions are made in an Excel workbook and saved to BigQuery with a reason.

| | |
|---|---|
| **When to run** | After loading new media (02) or new reference files (03), and whenever a film detail needs fixing |
| **How long** | About 1 minute per run |
| **What it changes** | Only the tables `title` and `title_alias`, and only when `DRY_RUN = False`. Every change (and every rejected suggestion) is saved in `title_history` with who, when and why. |
| **Safe to re-run?** | Yes. An old workbook can never overwrite newer decisions: changed rows are skipped. |
| **Automatic rule** | Films released before 2023 are set to **In MMM = no** (Home Entertainment ran PVOD media from 2023). Step 3 lists them; they are saved with the next run where `DRY_RUN = False`. |

Workbooks live in a Domino dataset. To get there: open this project's **Data** page → **Datasets** → **nbcu_mmm_dev**, then click through **us_hment_mmm → review → titles**. It has three folders:

| Folder | What is in it |
|---|---|
| `issued` | New workbooks for you to download |
| `returned` | Where you upload a filled-in workbook |
| `imported` | Workbooks already saved (moved here automatically) |

### How to use it
1. **Get a workbook.** With `returned` empty, click **Run ▸ Run All Cells**. Step 2 shows the new file name in `issued`.
2. **Fill it in.** Download it from `issued` (Step 2 shows exactly where; click the file name to download). In Excel, work through the numbered tabs. Each tab starts with a blue box that says what the problem is and what each choice does. Fill in only the yellow columns, with a **Reason** for every row you decide. Save it as .xlsx.
3. **Upload it to `returned`** with the **Upload files** button on that folder's page, then **Run All** (a check only). Read **Step 3** (every change, old → new) and **Step 4** (rows not applied). If something needs fixing, fix it in Excel and upload it again.
4. **Save.** If Step 3 is right, set `DRY_RUN = False` and **Run All**. The changes are saved and the workbook moves to `imported`. Set `DRY_RUN = True` again afterwards.

## Step 1 · Settings

In [ ]:
# True  = check only, nothing is changed (always start here)
# False = save the decisions in the returned workbook
DRY_RUN = True

## Step 2 · Run

In [ ]:
from IPython.display import Markdown, display

from us_hment_mmm.reference.review import (auto_rule_table, changes_table, notes_markdown, run,
                                           status_markdown)

try:
    result = run(dry_run=DRY_RUN)
except Exception as exc:
    display(Markdown(f"### 🛑 STOP — the review could not run\n\n`{exc}`\n\nNothing was "
                     "changed. Save this notebook and send it to the pipeline owner."))
    raise

display(status_markdown(result))

## Step 3 · Changes
Every value that will change (check only) or that was changed (`DRY_RUN = False`).

In [ ]:
display(changes_table(result['changes']))
display(auto_rule_table(result))

## Step 4 · Rows not applied
Rows whose item is no longer on the list, or whose values changed after the workbook was issued. They will be on the next workbook if they still need a decision.

In [ ]:
display(notes_markdown(result['skipped'], 'Every decided row was used.'))

---
This notebook needs a person to make the decisions, so it is not meant to run as a Domino Job.